### 1.1.9. Predefined Sets and Interval Expressions

AMPL provides two closely related mechanisms for referring to common numeric collections without
enumerating every member explicitly:

1. **Predefined infinite sets** — built-in names (`Integers`, `Reals`, `Binary`, `ASCII`, `EBCDIC`, `Display`)
   that represent universal numeric or string domains. These are used in `within`, `in`, and
   `ordered by` phrases of declarations; they cannot be iterated directly in an indexing expression.

2. **Arithmetic range (interval) expressions** — the `..` and `.. by` operators, which construct
   finite sets of numbers as arithmetic progressions. A range expression such as `1..10` or
   `2010..2040 by 5` is a valid set expression anywhere a set is expected.

Together these tools let a modeler write compact, data-driven index sets without listing every
period, scenario, or level number explicitly.

In [ ]:
# -----------------------------------------------------------------------
# ARITHMETIC RANGE EXPRESSION SYNTAX
# -----------------------------------------------------------------------
#
#   expr1 .. expr2              step 1 (default)
#   expr1 .. expr2 by expr3     explicit step
#
# Grammar (from AMPL Reference Manual A.6):
#
#   sexpr ::= expr .. expr [ by expr ]
#
# The resulting set contains the numbers:
#
#   e1,  e1 + e3,  e1 + 2*e3,  ...,  e1 + floor((e2-e1)/e3) * e3
#
# where e1 = lower bound, e2 = upper bound, e3 = step (default 1).
# All three are arbitrary arithmetic expressions (may reference parameters).
#
# -----------------------------------------------------------------------
# PREDEFINED INFINITE SETS
# -----------------------------------------------------------------------
#
#   Integers   -- all integers, in numeric order (infinite; not iterable)
#   Reals      -- all floating-point numbers, in numeric order (infinite; not iterable)
#   Binary     -- equivalent to {0, 1}  (same as 0..1 by 1; iterable)
#   ASCII      -- universal string/number set, ordered by ASCII collating sequence
#   EBCDIC     -- universal string/number set, ordered by EBCDIC collating sequence
#   Display    -- universal string/number set, ordered as in AMPL's display command
#
# Role: used in 'within', 'in', 'ordered by [reversed]', 'circular by [reversed]'
# phrases of set and param declarations.  May NOT appear as the iterated set
# in an indexing expression such as {i in Integers} (infinite — no bound).
# -----------------------------------------------------------------------

#### Arithmetic Range Expressions: `..` and `.. by`

The `..` operator is the primary way to build a finite set of numbers in AMPL. Its general form is:

```
e1 .. e2
e1 .. e2 by e3
```

where `e1`, `e2`, and `e3` are arithmetic expressions (literals, parameters, or formulas).

**Membership formula.** The set contains the numbers

$$e_1,\; e_1 + e_3,\; e_1 + 2 e_3,\; \ldots,\; e_1 + \left\lfloor \frac{e_2 - e_1}{e_3} \right\rfloor e_3$$

Each value is rounded to the nearest representable floating-point number. The default step is 1.

**When the step is negative**, `e3 < 0`, the sequence counts downward: the set contains
`e1, e1 + e3, ...` as long as the value is `>= e2`. The same floor formula applies.

**Ordering status.** A range expression used **inline** (e.g., in `{t in 1..T}` or in a `display`
command) is treated by AMPL as a *manifestly ordered* set — position-based functions such as
`first`, `last`, `prev`, and `next` apply to it. However, a **named set** defined by a range
expression (`set YEARS = 2010..2040 by 5;`) is **unordered** unless the declaration explicitly
adds `ordered`. See [1.1.5. Unordered Sets](<1_1_5 unordered_sets.ipynb>) and
[1.1.6. Ordered Sets] for the full treatment of ordering.

In [ ]:
# Range expression variants — every form is a valid set expression

# 1. Closed range, step 1 (default)
#    Members: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
display 1..10;

# 2. Closed range, explicit step
#    Members: {2010, 2015, 2020, 2025, 2030, 2035, 2040}
display 2010..2040 by 5;

# 3. Odd integers in [1, 10]
#    Members: {1, 3, 5, 7, 9}
display 1..10 by 2;

# 4. Counting downward (negative step)
#    Members: {10, 8, 6, 4, 2}
display 10..2 by -2;

# 5. Single-element range (lower bound = upper bound)
#    Members: {5}
display 5..5;

# 6. Empty range (lower bound > upper bound with default positive step)
#    Members: {} (zero elements)
display 10..1;    # empty — no members satisfy the condition

#### Predefined Infinite Sets

AMPL defines several built-in set names that represent infinite or universal collections.
They appear in declaration phrases — they cannot serve as the iterated domain in an indexing
expression because they are infinite and AMPL cannot enumerate their members.

| Name | Members | Ordering | Primary use |
|---|---|---|---|
| `Integers` | All integers (…, -2, -1, 0, 1, 2, …) | Numeric | `within Integers`; constraining integer-valued params/sets |
| `Reals` | All floating-point numbers | Numeric | `within Reals`; rarely needed (numerics default to Reals) |
| `Binary` | `{0, 1}` (finite — iterable) | Numeric | `within Binary`; equivalent to `binary` keyword on params |
| `ASCII` | All strings and numbers | ASCII collating sequence | `ordered by ASCII`; alphabetical ordering of string sets |
| `EBCDIC` | All strings and numbers | EBCDIC collating sequence | `ordered by EBCDIC`; mainframe/legacy ordering |
| `Display` | All strings and numbers | Numbers first (numeric), then strings (ASCII) | `ordered by Display`; matches AMPL display command ordering |

**`Integers` and `Reals`** are conceptual supersets used to restrict numeric domains.
Writing `within Integers` in a set declaration asserts that every member of the set must be an
integer-valued number. This is equivalent in effect to adding the `integer` keyword to each
numeric parameter that uses that set as its index domain.

**`Binary`** is the only predefined numeric set that is finite (`{0, 1}`) and therefore iterable.
It is equivalent to the arithmetic range `0..1 by 1`. Using `binary` as a param attribute is
the idiomatic AMPL form; `within Binary` and `in Binary` are equivalent alternatives.

**`ASCII`, `EBCDIC`, `Display`** are used exclusively with `ordered by` (or `circular by`) to
impose a specific collating sequence on string sets, regardless of the order in which data was
supplied. They represent the universal set of all strings and numbers.

In [ ]:
# Predefined sets in declaration phrases

# 'within Integers': every member of the set must be integer-valued
set PERIODS within Integers;

# Equivalent: the 'integer' attribute on an indexed parameter
param T integer > 0;
set WEEKS = 1..T;        # T must be integer; WEEKS inherits integer members

# 'within Binary': identical to the 'binary' param attribute
param active {PERIODS} within Binary;   # active[t] in {0,1}
# Equivalent:
param active2 {PERIODS} binary;         # same restriction, shorter syntax

# 'ordered by ASCII': alphabetical ordering of a string set
set PROD ordered by ASCII;   # members displayed and iterated in ASCII order

# 'ordered by Display': AMPL display-command ordering (numbers first, then strings)
set NUTR ordered by Display;

# 'within Reals': explicit (rarely needed; params are numeric by default)
param cost {PROD} within Reals, >= 0;   # same as just >= 0

#### Named Sets vs. Inline Range Expressions: Ordering Distinction

A subtle but important point in AMPL: a range expression used **inline** in an indexing
expression is treated as manifestly ordered, but the same expression used to **define a named
set** does not automatically make that set ordered.

| Form | Ordered? | Position functions valid? |
|---|---|---|
| `{t in 1..T}` inline in constraint | Yes (manifestly ordered) | `prev(t, 1..T)` valid |
| `set T_SET = 1..T;` without `ordered` | No (unordered by default) | `first(T_SET)` is a **model error** |
| `set T_SET ordered = 1..T;` | Yes | `first(T_SET)`, `prev(t, T_SET)` valid |

This asymmetry exists because a set declaration is a general-purpose entity whose ordering
is governed by its declaration attributes, not by how its value happens to be expressed.
The `ordered` keyword must appear explicitly in the declaration for AMPL to permit
position-dependent operations on the named set.

In practice, if a time-period set needs `prev` or `first`, declare it `ordered`:
```ampl
set PERIODS ordered = 1..T;
```

In [ ]:
# Ordering distinction: inline range vs. named set

param T integer > 0;
param avail {1..T} >= 0;   # inline range: valid; 1..T is a manifestly ordered set

# Named set WITHOUT 'ordered' — range defines the members but ordering is NOT inherited
set WEEKS = 1..T;          # unordered

# This constraint uses an inline range — 'prev' is valid because 1..T is manifestly ordered
var Make >= 0;
var Inv {0..T} >= 0;

# CORRECT: inline 1..T is manifestly ordered; t-1 arithmetic (not 'prev') is idiomatic here
# subject to Balance {t in 1..T}: Make + Inv[t-1] = Inv[t];

# Named set WITH 'ordered' — position functions are valid
set WEEKS2 ordered = 1..T;  # ordered; first/last/prev/next are valid
# first(WEEKS2)            -> first member (= 1 when T >= 1)
# last(WEEKS2)             -> last member  (= T)
# prev(t, WEEKS2)          -> member before t in the sequence

# WRONG: calling first() on the unordered named set is a model error
# display first(WEEKS);    # ERROR: 'first' applied to unordered set WEEKS

---
#### Example 1 — Minimal Arithmetic Ranges (Inline and Named)

This example shows the most common patterns: an inline range as a parameter index,
a named range set, and arithmetic on the dummy index.

In [ ]:
# --- Example 1: Inline and named arithmetic ranges ---

param T integer > 0;         # number of time periods (supplied in data)

# Parameters indexed directly over an inline range
param avail {1..T} >= 0;     # hours available in period t
param demand {1..T} >= 0;    # demand in period t

# Named range set — useful when the same period set is referenced many times
set WEEKS = 1..T;            # same members as 1..T; reuse avoids repetition

param rate >= 0;             # production rate (tons/hour)

var Make {1..T} >= 0;        # tons produced in period t

maximize TotalMake:
    sum {t in 1..T} Make[t];

# The dummy index t is numeric, so arithmetic t-1 is valid here
subject to Capacity {t in 1..T}:
    Make[t] <= avail[t] * rate;

subject to MeetDemand {t in 1..T}:
    Make[t] >= demand[t];

# Data section: T = 3, avail and demand supplied as vectors
data;
param T := 3;
param avail := 1 40  2 32  3 40;
param demand := 1 25  2 30  3 20;
param rate := 5;

---
#### Example 2 — Parameterized Range with `start`, `end`, and `by`

When the bounds or step of a range are model parameters, the set is recomputed automatically
whenever those parameter values change. This is the recommended practice: hard-coded year
literals in the model are brittle; parameterized bounds keep the model instance-agnostic.

In [ ]:
# --- Example 2: Fully parameterized range ---
# Fourer, Gay, Kernighan (AMPL Book, Chapter 5): parameterized year ranges

param start    integer;              # first year (supplied in data)
param end      > start integer;     # last year
param interval > 0 integer;         # step between years

set YEARS = start .. end by interval;  # recomputed when any of the three params change

param revenue {YEARS} >= 0;          # revenue in each year
param cost    {YEARS} >= 0;          # cost in each year

# Sum over the named set (equivalent to summing over start..end by interval inline)
maximize NetRevenue:
    sum {y in YEARS} (revenue[y] - cost[y]);

# Data: gives YEARS = {2010, 2015, 2020, 2025, 2030, 2035, 2040}
data;
param start    := 2010;
param end      := 2040;
param interval := 5;
param revenue  := 2010 100  2015 120  2020 130  2025 140  2030 150  2035 160  2040 170;
param cost     := 2010  80  2015  90  2020  95  2025 100  2030 105  2035 110  2040 115;

---
#### Example 3 — Predefined Sets in Parameter and Set Declarations

This example shows all three numeric predefined sets (`Integers`, `Reals`, `Binary`) used in
their canonical roles in a scheduling model. Each declaration constraint is enforced by AMPL
at data-load time.

In [ ]:
# --- Example 3: Predefined sets in declarations ---

set JOBS;                              # set of jobs (string-valued, supplied in data)
set MACHINES;                          # set of machines

# 'within Integers': every supplied member of PERIODS must be an integer
set PERIODS within Integers;           # e.g., {1, 2, 3, 4}

# 'within Binary': each param value must be 0 or 1
param assigned {JOBS, MACHINES} within Binary;
# Equivalent alternative using the 'binary' keyword:
param assigned2 {JOBS, MACHINES} binary;

# 'integer' keyword on a param: same effect as 'within Integers' applied to a scalar
param horizon integer > 0;            # total number of periods

param duration {JOBS} integer >= 1;   # job duration in periods (must be a positive integer)
param release  {JOBS} within Integers, >= 0;  # earliest start period
param deadline {j in JOBS} within Integers, >= release[j] + duration[j];

# Parameters over JOBS x PERIODS: processing time is non-negative real
param proctime {JOBS, PERIODS} >= 0;  # 'within Reals' not needed — params are real by default

# 'ordered by ASCII': products listed alphabetically in display output
set JOBS_SORTED ordered by ASCII;     # re-ordered view; same members as JOBS

---
#### Example 4 — Combining Range Expressions with Set Operators

Range expressions produce ordinary set values. They can be combined with `union`, `diff`,
`inter`, and `symdiff` just like any other set expression. The most common patterns are:
- union of two ranges to form a non-contiguous index set
- diff to exclude specific periods from a range

In [ ]:
# --- Example 4: Range expressions combined with set operators ---

# Union of two non-overlapping ranges
# Members: {2010, 2015, 2020, 2025, 2030, 2035, 2040, 2045}
set Y1 = 2010..2040 by 5;
set Y2 = 2020..2045 by 5;

display Y1 union Y2;    # {2010, 2015, 2020, 2025, 2030, 2035, 2040, 2045}
display Y1 inter Y2;    # {2020, 2025, 2030, 2035, 2040}
display Y1 diff  Y2;    # {2010, 2015}
display Y1 symdiff Y2;  # {2010, 2015, 2045}

# Excluding specific members from a range using diff
# All periods except the first and last
param T integer > 0;
set INNER_PERIODS = 1..T diff {1, T};   # {2, 3, ..., T-1}

# Non-contiguous planning horizon: two phases with a gap
param T1 integer > 0;
param T2 integer > T1;
param gap_start integer > T1;
param gap_end   integer >= gap_start, < T2;

set PHASE1 = 1..T1;
set PHASE2 = gap_end+1 .. T2;
set ACTIVE_PERIODS = PHASE1 union PHASE2;   # excludes the gap

param demand {ACTIVE_PERIODS} >= 0;
var  Produce {ACTIVE_PERIODS} >= 0;

minimize TotalProd: sum {t in ACTIVE_PERIODS} Produce[t];
subject to MeetDemand {t in ACTIVE_PERIODS}: Produce[t] >= demand[t];

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Parameterize the bounds and step of a range expression rather than hard-coding literal values.
Write <code>set YEARS = start .. end by interval;</code> where <code>start</code>,
<code>end</code>, and <code>interval</code> are declared parameters. This keeps the model
file stable across instances: only the data file changes, and AMPL recomputes the set
automatically. Hard-coded literals like <code>2010..2040 by 5</code> require editing the
model file whenever the planning horizon changes.</div>

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Use <code>within Integers</code> (or the equivalent <code>integer</code> keyword) on every
parameter or set that must hold integer values. AMPL checks the constraint at data-load time
and reports a clear error message — e.g., <em>"param T = 2.5 is not an integer"</em> — before
any solve is attempted. This is preferable to relying on the solver to reject non-integer
data silently.</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Predefined infinite sets (<code>Integers</code>, <code>Reals</code>, <code>ASCII</code>,
<code>EBCDIC</code>, <code>Display</code>) cannot be used as the iterated domain in an
indexing expression. Writing <code>{i in Integers}</code> or
<code>sum {i in Reals} ...</code> is a model error because AMPL cannot enumerate an
infinite set. Use a range expression (<code>1..N</code>, <code>a..b by k</code>) or a
finite declared set as the iteration domain. Only <code>Binary</code> (which equals
<code>{0,1}</code>) is finite and iterable.</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Fractional steps in range expressions introduce floating-point rounding errors. The expression
<code>0 .. 1 by 0.1</code> does not reliably produce <code>{0, 0.1, 0.2, ..., 1.0}</code> —
the value nominally equal to 1.0 may be slightly above or below the representable limit,
and membership tests like <code>1.0 in 0..1 by 0.1</code> may fail. Stick to integer steps
whenever possible; if real steps are required, verify membership with a small tolerance
rather than exact equality.</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
A named set defined by a range expression is <strong>unordered</strong> unless its declaration
includes the <code>ordered</code> keyword. Calling <code>first(S)</code>, <code>last(S)</code>,
<code>prev(t, S)</code>, or <code>next(t, S)</code> on an unordered named set is a model error
reported at generation time. If position-based functions are needed, write
<code>set S ordered = a..b;</code> explicitly.</div>

#### amplpy Equivalents

In amplpy the most common workflow is to set the bounding parameters (`T`, `start`, `end`, etc.)
from Python and let AMPL compute the resulting range sets. Alternatively, the Python
`range()` built-in produces a list of integers that can be assigned directly to a declared
AMPL set.

| Task | amplpy pattern |
|---|---|
| Set `T` so AMPL computes `1..T` | `ampl.param["T"] = 10` |
| Assign `start`, `end`, `interval` for `start..end by interval` | `ampl.param["start"] = 2010; ampl.param["end"] = 2040; ampl.param["interval"] = 5` |
| Assign explicit integer range to a declared set | `ampl.set["PERIODS"] = list(range(1, T+1))` |
| Assign stepped integer range to a declared set | `ampl.set["YEARS"] = list(range(2010, 2041, 5))` |
| Read back the members of a range-defined set | `ampl.set["YEARS"].members()` |
| Evaluate `Binary` inline | `ampl.eval("display Binary;")` (displays `{0, 1}`) |
| Check integer compliance of a param | `ampl.get_value("T")` — AMPL enforces `integer` at data load |

In [ ]:
# amplpy: setting range bounds from Python
from amplpy import AMPL

ampl = AMPL()

# Pattern 1: set parameter T; AMPL evaluates the range 1..T
ampl.eval("""
    param T integer > 0;
    set PERIODS = 1..T;
    param avail {PERIODS} >= 0;
""")

ampl.param["T"] = 6
# AMPL now computes PERIODS = {1, 2, 3, 4, 5, 6} automatically
ampl.param["avail"] = {1: 40, 2: 32, 3: 40, 4: 40, 5: 36, 6: 40}
print(list(ampl.set["PERIODS"].members()))   # [1, 2, 3, 4, 5, 6]

# Pattern 2: assign an explicit Python range to a declared set
ampl.eval("""
    param N integer > 0 default 1;
    set LEVELS within Integers;
""")
N = 8
ampl.set["LEVELS"] = list(range(1, N + 1))   # [1, 2, ..., 8]

# Pattern 3: stepped range using Python range()
ampl.eval("set YEARS within Integers;")
ampl.set["YEARS"] = list(range(2010, 2041, 5))  # [2010, 2015, ..., 2040]
print(list(ampl.set["YEARS"].members()))  # [2010, 2015, 2020, 2025, 2030, 2035, 2040]

# Pattern 4: parameterized range — preferred for self-documenting models
ampl.eval("""
    param yr_start integer;
    param yr_end   integer > yr_start;
    param yr_step  integer > 0;
    set PLAN_YEARS = yr_start .. yr_end by yr_step;
""")
ampl.param["yr_start"] = 2020
ampl.param["yr_end"]   = 2050
ampl.param["yr_step"]  = 10
print(list(ampl.set["PLAN_YEARS"].members()))  # [2020, 2030, 2040, 2050]

---
#### Edge Cases

**Edge case 1: Empty range (lower bound exceeds upper bound with positive step).**

When the lower bound is greater than the upper bound and the step is positive, the floor
formula yields zero members: the range is empty. This is not an error. Any constraint or
parameter indexed over an empty range generates zero instances. The empty range is returned
as `{}` and is a valid set value throughout the model.

In [ ]:
# Edge case 1: empty range

param T integer > 0;

# If T = 0 were allowed, 1..T would be empty; we guard with T > 0 on the param declaration.
# For illustration, a range that is always empty given positive T:
set EMPTY_RANGE = T+1 .. T;   # lower > upper; step defaults to 1 -> {} always

# Verifying: card(EMPTY_RANGE) = 0, no error
display card(EMPTY_RANGE);    # 0

# Constraint indexed over an empty range generates no constraint instances
param avail {1..T} >= 0 default 0;
var  Make {1..T} >= 0;

subject to EmptyCon {t in EMPTY_RANGE}:  # zero constraints; model still valid
    Make[t] <= avail[t];

# --- Data ---
data;
param T := 3;

**Edge case 2: Range expression whose upper bound equals the lower bound.**

The formula `e1 + floor((e2-e1)/e3) * e3` with `e1 = e2` gives `e1 + 0 = e1`. The range
is a singleton `{e1}` regardless of the step value. This is correct and not an error.

In [ ]:
# Edge case 2: singleton range

# Explicit singleton
display 5..5;          # {5} — one member
display 5..5 by 3;     # {5} — step irrelevant when lower = upper

# Parameterized singleton (can arise when T = 1)
param T integer > 0;
set WEEKS = 1..T;

# When T = 1: WEEKS = {1}, card(WEEKS) = 1
# The Balance constraint below generates exactly one instance:
param inv0 >= 0;
var  Inv {0..T} >= 0;
var  Make {1..T} >= 0;

subject to Balance {t in 1..T}:
    Make[t] + Inv[t-1] = Inv[t];

# t-1 with t=1 gives Inv[0], which is valid since Inv is indexed over 0..T.
# The singleton range does not cause boundary issues here because the index
# arithmetic (t-1) is numeric, not position-based.

data;
param T  := 1;
param inv0 := 5;

**Edge case 3: Very large range set and memory limits.**

AMPL stores each member of a range set explicitly in memory. A range like `1..1e9` attempts
to allocate space for one billion integers. On most systems this exhausts available memory.
If the model needs to reason about a large but implicit range (e.g., all non-negative
integers up to some bound), use `within Integers` as a membership constraint on a set
whose members are supplied selectively, rather than materializing the full range.

In [ ]:
# Edge case 3: large range — memory risk

# RISKY: materializes up to 1,000 members — manageable
set SMALL = 1..1000;
display card(SMALL);   # 1000 — fine

# DANGEROUS: attempts to allocate 1e9 members — likely out of memory
# set HUGE = 1..1e9;   # do NOT run on typical hardware

# SAFE ALTERNATIVE: declare a sparse set with the integer constraint;
# supply only the members that are actually needed
set SPARSE_INT within Integers;   # no materialization; members come from data

# In a data file or amplpy:
#   set SPARSE_INT := 1 50 100 500 1000 100000;
# Only 6 members are stored, regardless of how large the integers are.

# NOTE: The 'within Integers' constraint still validates that all supplied
# members are integer-valued at data-load time.

**Edge case 4: Negative step with lower bound less than upper bound produces an empty set.**

The floor formula handles negative steps correctly. When `e3 < 0` and `e1 < e2`,
the quantity `(e2 - e1) / e3` is negative, and its floor is a negative integer, meaning
zero members are generated — the set is empty. Only when `e1 >= e2` does a negative step
produce members.

In [ ]:
# Edge case 4: negative step direction

# Descending range — requires lower >= upper for non-empty result
display 10..2 by -2;    # {10, 8, 6, 4, 2} — five members
display 10..1 by -3;    # {10, 7, 4, 1} — four members

# Negative step with lower < upper: empty
display 2..10 by -2;    # {} — floor((10-2)/(-2)) = floor(-4) = -4 < 0 -> zero steps

# Practical use: reversed period indexing
param T integer > 0;
# Terminal period first, initial period last (sometimes useful for backward induction)
set REVERSE_PERIODS = T..1 by -1;   # {T, T-1, ..., 2, 1}

---

See also:
- [1.1.1. Set Declaration vs. Definition](<1_1_1 Set Declaration vs. Definition.ipynb>) — `=` vs `default` vs data assignment
- [1.1.4. Set Operations](<1_1_4 set_operations.ipynb>) — `union`, `inter`, `diff`, `symdiff`, `cross`
- [1.1.5. Unordered Sets](<1_1_5 unordered_sets.ipynb>) — ordering status of range-defined sets
- AMPL Book Chapter 5 (Fourer, Gay, Kernighan) — set operations and arithmetic progressions
- AMPL Reference Manual A.6.3 — formal syntax for interval and infinite-set expressions